In [1]:
%matplotlib inline
import numpy as np
import os
import pandas as pd
import glob
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression
from scipy.cluster.hierarchy import linkage, dendrogram
import japanize_matplotlib

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.cm import get_cmap
import seaborn as sns

from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import StratifiedKFold

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import roc_curve, auc


In [27]:
tasks = ["task1","task2"]
Days = [14, 17, 20, 24, 27, 31, 34, 38]
savefolder = "../data/231115FPRresults/"
if not os.path.exists(savefolder):
    os.mkdir(savefolder)

def load_data(Day, task):
    if Day == 38:
        df_feature= pd.read_csv("../data/fftfeatureD14_D38/feature38.csv")
    elif Day == 34:
        df_feature= pd.read_csv("../data/fftfeatureD14_D38/feature34.csv")
    elif Day == 31:
        df_feature= pd.read_csv("../data/fftfeatureD14_D38/feature31.csv")
    elif Day == 27:
        df_feature= pd.read_csv("../data/fftfeatureD14_D38/feature27.csv")
    elif Day == 24:
        df_feature= pd.read_csv("../data/fftfeatureD14_D38/feature24.csv")
    elif Day == 20:
        df_feature= pd.read_csv("../data/fftfeatureD14_D38/feature20.csv")
    elif Day == 17:
        df_feature= pd.read_csv("../data/fftfeatureD14_D38/feature17.csv")
    elif Day == 14:
        df_feature= pd.read_csv("../data/fftfeatureD14_D38/feature14.csv")

    if task == "task20over":
        df_feature["FLAG"] = df_feature["score"] > 20
    elif task == "task10under":
        df_feature["FLAG"] = df_feature["score"] < 10
    elif task == "task15over":
        df_feature["FLAG"] = df_feature["score"] > 15
    return df_feature

def extractX(df):
    X = df.iloc[:,1:87].values
    return X

def X2df(X,df):
    _df = pd.DataFrame(X)
    _df["batchno"]=df["batchno"].values
    _df["day"]=df["day"].values
    _df["score"]=df["score"].values
    _df["FLAG"]=df["FLAG"].values
    return _df

def idx_of_the_nearest(data, value):
    idx = np.argmin(np.abs(np.array(data) - value))
    return idx

# test

In [26]:
df_feature = load_data(38,"task20over")
X = extractX(df_feature)
y = df_feature["FLAG"].values
dish = df_feature["dish"].values
#unique_publisher = np.unique(dish)
dish_flag = df_feature[["dish","FLAG"]].drop_duplicates()
unique_publisher = dish_flag["dish"].values
flag_publisher = dish_flag["FLAG"].values

# 空のリストを作成（学習データの評価指標を入れる）
# 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5 の添え字を持つリストを作成
# それぞれの添え字に対応する要素に、学習データの評価指標を格納
# 例えば、fpr0.1は、fpr[0]に格納されている
fpr_designs = []
fpr_reals = []
acc_scores = []
pre_scores = []
rec_scores = []
f1_scores = []

kf = StratifiedKFold(n_splits=4, shuffle=True, random_state=100)
for fold, (tr_group_idx, va_group_idx) in enumerate(kf.split(unique_publisher, flag_publisher)):
    tr_groups, va_groups = unique_publisher[tr_group_idx], unique_publisher[va_group_idx]

    is_tr = np.isin(dish, tr_groups)
    is_va = np.isin(dish, va_groups)

    X_train, X_valid = X[is_tr], X[is_va]
    y_train, y_valid = y[is_tr], y[is_va]

    # Train
    #教師データをPCA
    pca = PCA()
    X_pca_train = pca.fit_transform(X_train)
    #教師データをPLS
    pls = PLSRegression(n_components=2)
    X_pls_train = pls.fit_transform(X_pca_train, y_train)
    X_dat_train = np.hstack((X_pls_train[0],X_pca_train[:,:10]))

    # Train model
    model = RandomForestClassifier()
    #model = XGBClassifier()
    model.fit(X_dat_train, y_train)

    # Validation
    # 各画像から特徴量抽出(pca+pls)
    X_pca_valid = pca.transform(X_valid)
    X_pls_valid = pls.transform(X_pca_valid)
    X_dat_valid = np.hstack((X_pls_valid,X_pca_valid[:,:10]))


    y_valid_pred = model.predict(X_dat_valid)
    y_proba = model.predict_proba(X_dat_valid)
    fpr, tpr, thresholds = roc_curve(y_true=y_valid, y_score=y_proba[:,1])

    # fpr0.1~0.5を0.05刻みでfprとtprとthresholdを出力
    for desired_fpr in [0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5]:
        idx = idx_of_the_nearest(fpr, desired_fpr)
        threshold = thresholds[idx]
        y_pred_desired_fpr = [1 if x >= threshold else 0 for x in y_proba[:,1]]
        acc = accuracy_score(y_valid, y_pred_desired_fpr)
        pre = precision_score(y_valid, y_pred_desired_fpr)
        rec = recall_score(y_valid, y_pred_desired_fpr)
        f1 = f1_score(y_valid, y_pred_desired_fpr)

        fpr_designs.append(desired_fpr)
        fpr_reals.append(fpr[idx])
        acc_scores.append(acc)
        pre_scores.append(pre)
        rec_scores.append(rec)
        f1_scores.append(f1)
# 結果をデータフレームに格納
result = pd.DataFrame({ "fpr_design": fpr_designs,
                        "fpr_real": fpr_reals,
                        "accuracy": acc_scores,
                        "precision": pre_scores,
                        "recall": rec_scores,
                        "f1": f1_scores})
# fpr_design毎に平均値を算出
result_mean = result.groupby("fpr_design").mean().reset_index()



# Do experiments

In [30]:
def do_cv(df_feature, FOLD, seed, textlog, logging):
    X = extractX(df_feature)
    y = df_feature["FLAG"].values
    dish = df_feature["dish"].values
    #unique_publisher = np.unique(dish)
    dish_flag = df_feature[["dish","FLAG"]].drop_duplicates()
    unique_publisher = dish_flag["dish"].values
    flag_publisher = dish_flag["FLAG"].values

    # 空のリストを作成（学習データの評価指標を入れる）
    # 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5 の添え字を持つリストを作成
    # それぞれの添え字に対応する要素に、学習データの評価指標を格納
    # 例えば、fpr0.1は、fpr[0]に格納されている
    fpr_designs = []
    fpr_reals = []
    acc_scores = []
    pre_scores = []
    rec_scores = []
    f1_scores = []

    kf = StratifiedKFold(n_splits=FOLD, shuffle=True, random_state=seed)
    if logging:
        textlog.write("seed:"+str(seed)+"\n")
    
    
    for fold, (tr_group_idx, va_group_idx) in enumerate(kf.split(unique_publisher, flag_publisher)):

        #データを分割
        tr_groups, va_groups = unique_publisher[tr_group_idx], unique_publisher[va_group_idx]
        if logging:
            textlog.write("KFOLD:CV="+str(fold)+"\n")
            textlog.write("train:\n")
            textlog.write(str(tr_groups)+"\n")
            textlog.write("valid:\n")
            textlog.write(str(va_groups)+"\n")
        
        
        is_tr = np.isin(dish, tr_groups)
        is_va = np.isin(dish, va_groups)

        X_train, X_valid = X[is_tr], X[is_va]
        y_train, y_valid = y[is_tr], y[is_va]

        # Train
        #教師データをPCA
        pca = PCA()
        X_pca_train = pca.fit_transform(X_train)
        #教師データをPLS
        pls = PLSRegression(n_components=2)
        X_pls_train = pls.fit_transform(X_pca_train, y_train)
        X_dat_train = np.hstack((X_pls_train[0],X_pca_train[:,:10]))

        # Train model
        model = RandomForestClassifier()
        #model = XGBClassifier()
        model.fit(X_dat_train, y_train)

        # Validation
        # 各画像から特徴量抽出(pca+pls)
        X_pca_valid = pca.transform(X_valid)
        X_pls_valid = pls.transform(X_pca_valid)
        X_dat_valid = np.hstack((X_pls_valid,X_pca_valid[:,:10]))


        y_valid_pred = model.predict(X_dat_valid)
        y_proba = model.predict_proba(X_dat_valid)
        fpr, tpr, thresholds = roc_curve(y_true=y_valid, y_score=y_proba[:,1])

        # fpr0.1~0.5を0.05刻みでfprとtprとthresholdを出力
        for desired_fpr in [0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5]:
            idx = idx_of_the_nearest(fpr, desired_fpr)
            threshold = thresholds[idx]
            y_pred_desired_fpr = [1 if x >= threshold else 0 for x in y_proba[:,1]]
            acc = accuracy_score(y_valid, y_pred_desired_fpr)
            pre = precision_score(y_valid, y_pred_desired_fpr)
            rec = recall_score(y_valid, y_pred_desired_fpr)
            f1 = f1_score(y_valid, y_pred_desired_fpr)

            fpr_designs.append(desired_fpr)
            fpr_reals.append(fpr[idx])
            acc_scores.append(acc)
            pre_scores.append(pre)
            rec_scores.append(rec)
            f1_scores.append(f1)
    # 結果をデータフレームに格納
    result = pd.DataFrame({ "fpr_design": fpr_designs,
                            "fpr_real": fpr_reals,
                            "accuracy": acc_scores,
                            "precision": pre_scores,
                            "recall": rec_scores,
                            "f1": f1_scores})
    # fpr_design毎に平均値を算出
    result_mean = result.groupby("fpr_design").mean().reset_index()
    # seedを追加
    result_mean["seed"] = seed

    if logging:
        textlog.write("\n")

    return result_mean

def do_experiments(task, Days, n_exp, FOLD=4,savefolder = savefolder):
    results = pd.DataFrame()
    # make folder
    if not os.path.exists(savefolder+task):
        os.mkdir(savefolder+task)
    textlog = open(savefolder+task+"/FOLD_"+str(FOLD)+"_n="+str(n_exp)+'_logfile.txt', 'w', encoding='UTF-8')
    for i in range(n_exp):
        for Day in Days:
            df_feature = load_data(Day,task)
            if Day == 38:
                log = True
            else:
                log = False
            _results = do_cv(df_feature, FOLD=FOLD, seed=i+100, textlog=textlog, logging = log)
            _results["Day"] = Day
            results = pd.concat([results,_results])
    textlog.close()
    results.to_csv(savefolder+task+"/FOLD_"+str(FOLD)+"_n="+str(n_exp)+"_result_bugfixed"+".csv")
    return results

In [32]:
results20 = do_experiments("task20over", Days, 50)
results10 = do_experiments("task10under", Days, 50)
# it take about 4 hours

In [117]:
def caluculate_models_scores(results_data, task):
    # mean and sd
    results_mean = results_data.groupby(["fpr_design","Day"])[["fpr_real","accuracy","precision","recall","f1"]].mean().reset_index()
    results_std = results_data.groupby(["fpr_design","Day"])[["fpr_real","accuracy","precision","recall","f1"]].std().reset_index()
    # dayからfpr_designの列だけ取得
    results_meta = results_mean[["fpr_design","Day"]]
    # fpr_realからf1を有効数字2桁で丸めて文字型に変換して表示
    results_cal = results_mean[["fpr_real","accuracy","precision","recall","f1"]] .round(3).astype(str)+  "+-" + results_std[["fpr_real","accuracy","precision","recall","f1"]].round(3).astype(str)
    # 結合
    results_bind = pd.concat([results_meta,results_cal],axis=1)
    # save
    for focus_fpr in [0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5]:
        results_bind_focus = results_bind[results_bind["fpr_design"]==focus_fpr]
        results_bind_focus = results_bind_focus.drop(columns=["fpr_design"])
        # raw save
        # results_bind_focus.to_csv(savefolder+task+"/result_fpr_"+str(focus_fpr)+".csv")
        # crosstab
        results_bind_focus = results_bind_focus.set_index("Day")
        results_bind_focus = results_bind_focus.T
        results_bind_focus = results_bind_focus.rename(columns={14:"Day 14",17:"Day 17",20:"Day 20",24:"Day 24",27:"Day 27",31:"Day 31",34:"Day 34",38:"Day 38"})
        results_bind_focus = results_bind_focus.rename(index={"fpr_real":"FPR","accuracy":"Accuracy","precision":"Precision","recall":"Recall","f1":"F1"})
        results_bind_focus.index.name = f"FP rate={focus_fpr}"
        results_bind_focus.to_csv(savefolder+task+"/result_fpr_"+str(focus_fpr)+".csv")


In [118]:
caluculate_models_scores(results20, "task20over")

In [119]:
caluculate_models_scores(results10, "task10under")